# Platform Monitoring Queries

This notebook answers questions by analyzing pipeline execution metadata stored in the `pipeline_executions` Delta table. 

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("AnalyzePipelineMonitoring")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_EXECUTIONS_PATH = (
    "s3a://delta-data/metadata/pipeline_executions"
)

executions = (
    spark.read
    .format("delta")
    .load(PIPELINE_EXECUTIONS_PATH)
)

executions.createOrReplaceTempView("pipeline_executions")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 19:11:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/26 19:11:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Questions

### Which dataset fails validation most frequently?

In [2]:
spark.sql("""
    SELECT
        dataset,
        SUM(CASE WHEN validation_failure_count > 0 THEN 1 ELSE 0 END)
            AS executions_with_failures,
        SUM(COALESCE(rejected_records, 0))
            AS total_rejected_records,
        SUM(COALESCE(validation_failure_count, 0))
            AS total_rule_violations
    FROM pipeline_executions
    WHERE stage = 'silver'
    GROUP BY dataset
    ORDER BY executions_with_failures DESC, total_rejected_records DESC;
""").show(truncate=False)

26/09/26 19:12:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------------+----------------------+---------------------+
|dataset    |executions_with_failures|total_rejected_records|total_rule_violations|
+-----------+------------------------+----------------------+---------------------+
|taxi_trips |1                       |194717                |738006               |
|taxi_zones |1                       |1                     |1                    |
|weather    |0                       |0                     |0                    |
|air_quality|0                       |0                     |0                    |
+-----------+------------------------+----------------------+---------------------+



### Which dataset requires the longest processing time?

In [3]:
spark.sql("""
    SELECT
        stage,
        dataset,
        COUNT(*) AS execution_count,
        ROUND(AVG(duration_seconds), 3) AS average_duration_seconds,
        ROUND(MAX(duration_seconds), 3) AS longest_duration_seconds
    FROM pipeline_executions
    GROUP BY stage, dataset
    ORDER BY average_duration_seconds DESC
""").show(truncate=False)

+-----------------+--------------------------------+---------------+------------------------+------------------------+
|stage            |dataset                         |execution_count|average_duration_seconds|longest_duration_seconds|
+-----------------+--------------------------------+---------------+------------------------+------------------------+
|gold             |integrated_taxi_trips           |1              |64.604                  |64.604                  |
|silver           |weather                         |1              |33.05                   |33.05                   |
|silver           |air_quality                     |1              |25.123                  |25.123                  |
|silver           |taxi_trips                      |1              |24.672                  |24.672                  |
|bronze           |air_quality                     |1              |12.598                  |12.598                  |
|bronze           |taxi_trips                   

### How many records were rejected during each execution?

In [4]:
spark.sql("""
    SELECT
        pipeline_run_id,
        execution_id,
        stage,
        dataset,
        rejected_records,
        started_at
    FROM pipeline_executions
    ORDER BY rejected_records DESC, started_at DESC
""").show(100, truncate=False)

+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|pipeline_run_id                     |execution_id                        |stage            |dataset                         |rejected_records|started_at                |
+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|a5244204-b416-423b-b14b-1ca4f777a8d5|d39a0eb3-30d8-40b3-9f3d-6fd48a7c50d8|silver           |taxi_trips                      |194717          |2026-09-26 18:43:15.723948|
|a5244204-b416-423b-b14b-1ca4f777a8d5|3a526daf-677b-4087-9c42-3f0eca4d5e95|silver           |taxi_zones                      |1               |2026-09-26 18:43:11.414977|
|a5244204-b416-423b-b14b-1ca4f777a8d5|9127726a-85f8-49c0-9bed-7e2188f56722|analytics_refresh|zone_weather_demand_variation   |0               |20

### How has processing time changed over multiple executions?

In [5]:
spark.sql("""
    SELECT
        stage,
        dataset,
        pipeline_run_id,
        started_at,
        duration_seconds
    FROM pipeline_executions
    ORDER BY stage, dataset, started_at
""").show(100, truncate=False)

+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|stage            |dataset                         |pipeline_run_id                     |started_at                |duration_seconds|
+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|analytics_refresh|air_quality_vs_demand           |a5244204-b416-423b-b14b-1ca4f777a8d5|2026-09-26 18:45:58.250947|2.281           |
|analytics_refresh|average_trip_distance_by_weather|a5244204-b416-423b-b14b-1ca4f777a8d5|2026-09-26 18:45:53.87597 |3.131           |
|analytics_refresh|monthly_zone_demand             |a5244204-b416-423b-b14b-1ca4f777a8d5|2026-09-26 18:45:48.936536|4.627           |
|analytics_refresh|zone_weather_demand_variation   |a5244204-b416-423b-b14b-1ca4f777a8d5|2026-09-26 18:46:00.705332|2.061           |
|bronze           |air_quality                     |a5244204-b

In [6]:
spark.stop()